In [24]:
# Make repaired datasets for all california jurisdictions
# SLOW!

import sys
import os
from dotenv import load_dotenv, find_dotenv
import matplotlib.pyplot as plt
import time
import pandas as pd
import numpy as np
import json
import glob

load_dotenv(find_dotenv())

ROOT_PATH = os.getenv("ROOT_PATH")
MY_DATA_PATH = os.getenv("MY_DATA_PATH")
RAW_DATA_PATH = os.getenv("RAW_DATA_PATH")
DEWEY_PATH = os.path.join(RAW_DATA_PATH, "dewey-downloads", "building-permits-united-states")

sys.path.append(os.path.join(ROOT_PATH, "scripts"))
import data_utils as du

sys.path.append(os.path.join(ROOT_PATH, "agent/scripts"))
from data_repair import data_repair, _slugify

SUMMARY_FILEPATH = os.path.join(MY_DATA_PATH, f"dewey_summary.parquet")

COLUMNS = [
    'PERMIT_NUMBER', 'JURISDICTION', 'STATE', 
    'FILE_DATE', 'PERMIT_DATE', 'FINAL_DATE', 
    'STATUS_NORMALIZED', 'STATUS_ORIGINAL', 
    'RECORD_TYPE_ORIGINAL', 'RECORD_SUBTYPE_ORIGINAL', 
    'APN', 'STREET', 'ZIPCODE', 'COUNTY_FIPS', 'CBSA_FIPS',
    'DESCRIPTION', 'DATA'
]  # Columns to load from data file

OUTPUT_COLS = [k for k in COLUMNS if k != 'DATA'] # Don't include DATA in output cols to save space
OUTPUT_COLS = OUTPUT_COLS + ['STATUS_NORMALIZED_FLAG', 'FILE_DATE_FLAG', 'PERMIT_DATE_FLAG', 'FINAL_DATE_FLAG', 'INFERRED_SCHEMA']

OUTPUT_FILEPATH = os.path.join(MY_DATA_PATH, "ca_quality_check.parquet")
REPORT_FILEPATH = os.path.join(ROOT_PATH, "reports", "2026-07-30-ca-quality-check.md")
CA_DATA_DIR = os.path.join(MY_DATA_PATH, "processed_data", "ca")


In [25]:
# Get parquet file list

files = glob.glob(os.path.join(CA_DATA_DIR, "*.parquet"))


In [26]:
# Compile quality check results

df = []
for f in files:
    city_df = pd.read_parquet(f)
    city_df['FILE_YEAR'] = city_df['FILE_DATE'].dt.year
    city_df['PERMIT_DATE_ONLY'] = (city_df['PERMIT_DATE'].notna()) & (city_df['FINAL_DATE'].isna())
    city_df['FINAL_DATE_ONLY'] = (city_df['FINAL_DATE'].notna()) & (city_df['PERMIT_DATE'].isna())
    city_df['PERMIT_AND_FINAL_DATE'] = (city_df['PERMIT_DATE'].notna()) & (city_df['FINAL_DATE'].notna())
    dfg = city_df.groupby(
        ['JURISDICTION', 'STATE', 'STATUS_NORMALIZED', 'FILE_YEAR'],
        dropna=False
    ).agg(
        COUNT = ('JURISDICTION', 'count'),
        PERMIT_DATE_ONLY = ('PERMIT_DATE_ONLY', 'sum'),
        FINAL_DATE_ONLY = ('FINAL_DATE_ONLY', 'sum'),
        PERMIT_AND_FINAL_DATE = ('PERMIT_AND_FINAL_DATE', 'sum')
    ).reset_index()
    df.append(dfg)

df = pd.concat(df).sort_values(
    by=['STATE', 'JURISDICTION', 'STATUS_NORMALIZED', 'FILE_YEAR'],
    ascending=True
).reset_index(drop=True)



In [27]:
# Preamble for report

num_jurisdictions = len(df[['JURISDICTION', 'STATE']].drop_duplicates())

MD = f"""
# Data quality report for California jurisidictions

This report checks for data quality of 4 variables:
- STATUS_NORMALIZED (Active, Final, In Review, Inactive)
- FILE_DATE
- PERMIT_DATE
- FINAL_DATE

There are {num_jurisdictions} total jurisidctions.

The quality check is performed after data repair rescripts are run for each jurisdiction. The data repair scripts check the provided data fields against the included raw JSON.

"""

In [28]:
# Report on FILE_DATE missingness by jurisdiction

FILE_DATE_THRESHOLD = 0.2

left_df = df.groupby(['JURISDICTION', 'STATE']).agg(COUNT = ('COUNT', 'sum')).reset_index()
right_df = df.loc[df['FILE_YEAR'].isna()].groupby(['JURISDICTION', 'STATE']).agg(FILE_DATE_MISSING = ('COUNT', 'sum')).reset_index()
file_date_df = left_df.merge(right_df, on=['JURISDICTION', 'STATE'], how='inner')

file_date_df['FILE_DATE_MISSING_RATE'] = file_date_df['FILE_DATE_MISSING'] / file_date_df['COUNT']
file_date_df['FILE_DATE_BAD'] = file_date_df['FILE_DATE_MISSING_RATE'] > FILE_DATE_THRESHOLD

num_file_date_bad = file_date_df['FILE_DATE_BAD'].sum()

share_file_date_bad = file_date_df.loc[file_date_df['FILE_DATE_BAD'], 'COUNT'].sum() / file_date_df['COUNT'].sum()

file_date_bad_jurisdictions = ", ".join(file_date_df.loc[file_date_df['FILE_DATE_BAD'], 'JURISDICTION'].unique().tolist())

MD += f"""
## FILE_DATE 

A jurisdiction was considered unusable if more than {FILE_DATE_THRESHOLD:.0%} of the permits had missing file dates.

{num_file_date_bad} out of {num_jurisdictions} jurisdictions were considered unusable because of missing FILE_DATE.  

They account for {share_file_date_bad:.1%} of the total permits in the data.

The jurisdictions are: {file_date_bad_jurisdictions}

"""

In [29]:
# Report on STATUS_NORMALIZED missingness by jursidiction

STATUS_NORMALIZED_THRESHOLD = 0.2

left_df = df.groupby(['JURISDICTION', 'STATE']).agg(COUNT = ('COUNT', 'sum')).reset_index()
right_df = df.loc[df['STATUS_NORMALIZED'].isna()].groupby(['JURISDICTION', 'STATE']).agg(STATUS_NORMALIZED_MISSING = ('COUNT', 'sum')).reset_index()
status_normalized_df = left_df.merge(right_df, on=['JURISDICTION', 'STATE'], how='inner')

status_normalized_df['STATUS_NORMALIZED_MISSING_RATE'] = status_normalized_df['STATUS_NORMALIZED_MISSING'] / status_normalized_df['COUNT']
status_normalized_df['STATUS_NORMALIZED_BAD'] = status_normalized_df['STATUS_NORMALIZED_MISSING_RATE'] > STATUS_NORMALIZED_THRESHOLD

num_status_normalized_bad = status_normalized_df['STATUS_NORMALIZED_BAD'].sum()

share_status_normalized_bad = status_normalized_df.loc[status_normalized_df['STATUS_NORMALIZED_BAD'], 'COUNT'].sum() / status_normalized_df['COUNT'].sum()

status_normalized_bad_jurisdictions = ", ".join(status_normalized_df.loc[status_normalized_df['STATUS_NORMALIZED_BAD'], 'JURISDICTION'].unique().tolist())

MD += f"""
## STATUS_NORMALIZED

A jurisdiction was considered unusable if more than {STATUS_NORMALIZED_THRESHOLD:.0%} of the permits had missing STATUS_NORMALIZED.

{num_status_normalized_bad} out of {num_jurisdictions} jurisdictions were considered unusable because of missing STATUS_NORMALIZED.

They account for {share_status_normalized_bad:.1%} of the total permits in the data.

The jurisdictions are: {status_normalized_bad_jurisdictions}

"""

In [30]:
# Timeline measurability concepts
#
# CONCEPT 1:  
# - Active permits must have FILE_DATE, PERMIT_DATE
# - Final permits must have FILE_DATE, PERMIT_DATE, FINAL_DATE
# 
# CONCEPT 2:
# - Active permits must have FILE_DATE, PERMIT_DATE
# - Final permits must have FILE_DATE, FINAL_DATE
#
# CONCEPT 3:
# - Active and final permits muat have FILE_DATE, PERMIT_DATE
#
# CONCEPT 4:
# - Active and Final permits must have FILE_DATE

THRESHOLD = 0.75

active_df = df.loc[(df['STATUS_NORMALIZED'] == 'Active') & (df['FILE_YEAR'].notna())]
final_df = df.loc[(df['STATUS_NORMALIZED'] == 'Final') & (df['FILE_YEAR'].notna())]

active_df['ACTIVE_CONCEPT_1'] = active_df['PERMIT_DATE_ONLY'] + active_df['PERMIT_AND_FINAL_DATE']
active_df['ACTIVE_CONCEPT_2'] = active_df['PERMIT_DATE_ONLY'] + active_df['PERMIT_AND_FINAL_DATE']
active_df['ACTIVE_CONCEPT_3'] = active_df['PERMIT_DATE_ONLY'] + active_df['PERMIT_AND_FINAL_DATE']
active_df['ACTIVE_CONCEPT_4'] = active_df['COUNT']

final_df['FINAL_CONCEPT_1'] = final_df['PERMIT_AND_FINAL_DATE']
final_df['FINAL_CONCEPT_2'] = final_df['FINAL_DATE_ONLY'] + final_df['PERMIT_AND_FINAL_DATE']
final_df['FINAL_CONCEPT_3'] = final_df['PERMIT_DATE_ONLY'] + final_df['PERMIT_AND_FINAL_DATE']
final_df['FINAL_CONCEPT_4'] = final_df['COUNT']

time_df = pd.merge(
    active_df[['JURISDICTION', 'STATE', 'FILE_YEAR', 'ACTIVE_CONCEPT_1', 'ACTIVE_CONCEPT_2', 'ACTIVE_CONCEPT_3', 'ACTIVE_CONCEPT_4']],
    final_df[['JURISDICTION', 'STATE', 'FILE_YEAR', 'FINAL_CONCEPT_1', 'FINAL_CONCEPT_2', 'FINAL_CONCEPT_3', 'FINAL_CONCEPT_4']],
    on=['JURISDICTION', 'STATE', 'FILE_YEAR'],
    how='outer'
)
for col in ['ACTIVE_CONCEPT_1', 'ACTIVE_CONCEPT_2', 'ACTIVE_CONCEPT_3', 'ACTIVE_CONCEPT_4', 'FINAL_CONCEPT_1', 'FINAL_CONCEPT_2', 'FINAL_CONCEPT_3', 'FINAL_CONCEPT_4']:
    time_df[col] = time_df[col].fillna(0)
time_df['COUNT'] = time_df['ACTIVE_CONCEPT_4'] + time_df['FINAL_CONCEPT_4']

for c in ['1', '2', '3', '4']:
    time_df[f'CONCEPT_{c}_USABLE'] = True
    time_df.loc[
        (time_df[f'ACTIVE_CONCEPT_{c}']/(time_df[f'ACTIVE_CONCEPT_4']+1e-6) < THRESHOLD),
        f'CONCEPT_{c}_USABLE'
    ] = False
    time_df.loc[
        (time_df[f'FINAL_CONCEPT_{c}']/(time_df[f'FINAL_CONCEPT_4']+1e-6) < THRESHOLD),
        f'CONCEPT_{c}_USABLE'
    ] = False

MD += f"""
## TIMELINE MEASURABILITY

We define four concepts of timeline measurability:

- CONCEPT 1:
    - Active permits must have FILE_DATE, PERMIT_DATE
    - Final permits must have FILE_DATE, PERMIT_DATE, FINAL_DATE
- CONCEPT 2:
    - Active permits must have FILE_DATE, PERMIT_DATE
    - Final permits must have FILE_DATE, FINAL_DATE
- CONCEPT 3:
    - Active and final permits must have FILE_DATE, PERMIT_DATE
- CONCEPT 4:
    - Active and final permits must have FILE_DATE

A jurisdiction/year is considered to be "usable" with a concept if more than {THRESHOLD:.0%} of the permits with a FILE_DATE in that year satisfy the requirements for the concept. (Note that Concept 4 is always usable when FILE_DATE is available.)

Below is a table showing, for each range of years, the number of jurisdictions with usable years for each concept for that entire year range.

"""

MD += "| Filing Year Range | "
for c in ['1', '2', '3', '4']:
    MD += f"CONCEPT_{c} | "
MD += "\n"
MD += "| --- | "
for c in ['1', '2', '3', '4']:
    MD += "--- | "
MD += "\n"

year_end = 2025
for year_start in range(2000, 2023):
    temp_df = time_df.loc[(time_df['FILE_YEAR'] >= year_start) & (time_df['FILE_YEAR'] <= year_end)]
    MD += f"| {year_start}-{year_end} | "
    for c in ['1', '2', '3', '4']:
        jur_df = temp_df.groupby(['JURISDICTION', 'STATE']).agg(
            USABLE_ALL = (f'CONCEPT_{c}_USABLE', 'all'),
        ).reset_index()
        num_usable_jur = jur_df['USABLE_ALL'].sum()
        MD += f"{num_usable_jur} | "
    MD += "\n"

In [31]:
# Write report to file

with open(REPORT_FILEPATH, 'w') as f:
    f.write(MD)